In [58]:
%pip install numpy 
%pip install pandas
import numpy as np
import pandas as pd


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
df=pd.read_csv(r"C:\Users\admin\Downloads\customer_shopping.csv")
df.head(10)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [60]:
df.describe(include='all')
df.isnull().sum()


Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [61]:
df["Review Rating"]=df.groupby("Category")["Review Rating"].transform(lambda x: x.fillna(x.median()))
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [62]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(" ","_")
df=df.rename(columns={"purchase_amount_(usd)":"purchase_amount"})

In [63]:
df["age"]
labels=['young_adult','adult','middle_aged','senior']
df['age_group']=pd.qcut(df["age"],q=4,labels=labels)
df["age_group"]


0       middle_aged
1       young_adult
2       middle_aged
3       young_adult
4       middle_aged
           ...     
3895          adult
3896    middle_aged
3897    middle_aged
3898          adult
3899    middle_aged
Name: age_group, Length: 3900, dtype: category
Categories (4, str): ['young_adult' < 'adult' < 'middle_aged' < 'senior']

In [ ]:
frequency_mapping={
    'Fortnightly':14,
    "Weekly":7,
    'Monthly':30,
    'Quarterly':90,
    'Annually':365,
    'Every_3_Months':90
}
df["frequency_of_purhase_days"]=df["frequency_of_purchases"].map(frequency_mapping)
df["frequency_of_purhase_days"].head()

In [65]:
(df["promo_code_used"]==df["discount_applied"]).all()
df=df.drop("promo_code_used",axis=1)

In [ ]:
%pip install  SQLALchemy PyMySQL
from sqlalchemy import create_engine

In [68]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine

username = "root"
password = quote_plus("sumit@3104")

host = "localhost"
port = 3306
database = "ar"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as conn:
    print("MySQL connected successfully!")

MySQL connected successfully!


In [69]:
df.to_sql(
    "customer_shopping",
    con=engine,
    if_exists="replace",
    index=False
)

print("Clean dataset uploaded successfully!")


Clean dataset uploaded successfully!


In [71]:
query = """
SELECT *
FROM ar.customer_shopping
LIMIT 10
"""

data = pd.read_sql(query, engine)

data

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,frequency_of_purhase_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,middle_aged,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,young_adult,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,middle_aged,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,young_adult,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,middle_aged,365.0
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,14,Venmo,Weekly,middle_aged,7.0
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,49,Cash,Quarterly,senior,90.0
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,19,Credit Card,Weekly,young_adult,7.0
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,8,Venmo,Annually,young_adult,365.0
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,middle_aged,90.0
